In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

# === INPUT PATHS ===
csv_folder_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/"
csv_input_name = "intensity_summary.csv"
csv_output_name = "summary_foldchange_pvalues.csv"

In [2]:
common_mz_df = pd.read_csv(os.path.join(csv_folder_path, csv_input_name))


In [3]:
df = common_mz_df.copy()

# 1️⃣ Compute mean metrics per common_group and group
grouped = df.groupby(['common_group', 'group']).agg(
    mean_max=('max_intensity', 'mean'),
    mean_mean=('mean_intensity', 'mean'),
    mean_max_tic=('max_tic_intensity', 'mean'),
    mean_mean_tic=('mean_tic_intensity', 'mean')
).reset_index()

# 2️⃣ Pivot to have groups as columns
pivot_max = grouped.pivot(index='common_group', columns='group', values='mean_max')
pivot_mean = grouped.pivot(index='common_group', columns='group', values='mean_mean')
pivot_max_tic = grouped.pivot(index='common_group', columns='group', values='mean_max_tic')
pivot_mean_tic = grouped.pivot(index='common_group', columns='group', values='mean_mean_tic')

# 3️⃣ Compute fold changes for individual groups
fold_changes = {
    'fc_max_aad/yad': pivot_max['aad'] / pivot_max['yad'],
    'fc_max_aad/ac': pivot_max['aad'] / pivot_max['ac'],
    'fc_max_yad/yc': pivot_max['yad'] / pivot_max['yc'],
    'fc_mean_aad/yad': pivot_mean['aad'] / pivot_mean['yad'],
    'fc_mean_aad/ac': pivot_mean['aad'] / pivot_mean['ac'],
    'fc_mean_yad/yc': pivot_mean['yad'] / pivot_mean['yc'],
    'fc_max_tic_aad/yad': pivot_max_tic['aad'] / pivot_max_tic['yad'],
    'fc_max_tic_aad/ac': pivot_max_tic['aad'] / pivot_max_tic['ac'],
    'fc_max_tic_yad/yc': pivot_max_tic['yad'] / pivot_max_tic['yc'],
    'fc_mean_tic_aad/yad': pivot_mean_tic['aad'] / pivot_mean_tic['yad'],
    'fc_mean_tic_aad/ac': pivot_mean_tic['aad'] / pivot_mean_tic['ac'],
    'fc_mean_tic_yad/yc': pivot_mean_tic['yad'] / pivot_mean_tic['yc'],
}

# 4️⃣ Compute fold changes for AD vs Control and Aged vs Young
# AD = aad + yad, Control = ac + yc
# Aged = yad + aad, Young = yc + ac
fold_changes['fc_max_AD/Control'] = (pivot_max['aad'] + pivot_max['yad']) / (pivot_max['ac'] + pivot_max['yc'])
fold_changes['fc_mean_AD/Control'] = (pivot_mean['aad'] + pivot_mean['yad']) / (pivot_mean['ac'] + pivot_mean['yc'])
fold_changes['fc_max_tic_AD/Control'] = (pivot_max_tic['aad'] + pivot_max_tic['yad']) / (pivot_max_tic['ac'] + pivot_max_tic['yc'])
fold_changes['fc_mean_tic_AD/Control'] = (pivot_mean_tic['aad'] + pivot_mean_tic['yad']) / (pivot_mean_tic['ac'] + pivot_mean_tic['yc'])

fold_changes['fc_max_Aged/Young'] = (pivot_max['aad'] + pivot_max['ac']) / (pivot_max['yad'] + pivot_max['yc'])
fold_changes['fc_mean_Aged/Young'] = (pivot_mean['aad'] + pivot_mean['ac']) / (pivot_mean['yad'] + pivot_mean['yc'])
fold_changes['fc_max_tic_Aged/Young'] = (pivot_max_tic['aad'] + pivot_max_tic['ac']) / (pivot_max_tic['yad'] + pivot_max_tic['yc'])
fold_changes['fc_mean_tic_Aged/Young'] = (pivot_mean_tic['aad'] + pivot_mean_tic['ac']) / (pivot_mean_tic['yad'] + pivot_mean_tic['yc'])

# 5️⃣ Compute p-values
def compute_pvals(df, group1_list, group2_list, col):
    common_groups = df['common_group'].unique()
    pvals = []
    for g in common_groups:
        vals1 = df[(df['common_group'] == g) & (df['group'].isin(group1_list))][col]
        vals2 = df[(df['common_group'] == g) & (df['group'].isin(group2_list))][col]
        if len(vals1) > 1 and len(vals2) > 1:
            p = ttest_ind(vals1, vals2, equal_var=False).pvalue
        else:
            p = np.nan
        pvals.append(p)
    return pd.Series(pvals, index=common_groups)

# Individual comparisons
pvals = {
    'pval_max_aad/yad': compute_pvals(df, ['aad'], ['yad'], 'max_intensity'),
    'pval_max_aad/ac': compute_pvals(df, ['aad'], ['ac'], 'max_intensity'),
    'pval_max_yad/yc': compute_pvals(df, ['yad'], ['yc'], 'max_intensity'),
    'pval_mean_aad/yad': compute_pvals(df, ['aad'], ['yad'], 'mean_intensity'),
    'pval_mean_aad/ac': compute_pvals(df, ['aad'], ['ac'], 'mean_intensity'),
    'pval_mean_yad/yc': compute_pvals(df, ['yad'], ['yc'], 'mean_intensity'),
    'pval_max_tic_aad/yad': compute_pvals(df, ['aad'], ['yad'], 'max_tic_intensity'),
    'pval_max_tic_aad/ac': compute_pvals(df, ['aad'], ['ac'], 'max_tic_intensity'),
    'pval_max_tic_yad/yc': compute_pvals(df, ['yad'], ['yc'], 'max_tic_intensity'),
    'pval_mean_tic_aad/yad': compute_pvals(df, ['aad'], ['yad'], 'mean_tic_intensity'),
    'pval_mean_tic_aad/ac': compute_pvals(df, ['aad'], ['ac'], 'mean_tic_intensity'),
    'pval_mean_tic_yad/yc': compute_pvals(df, ['yad'], ['yc'], 'mean_tic_intensity'),
}

# AD/Control and Aged/Young p-values
pvals['pval_max_AD/Control'] = compute_pvals(df, ['aad', 'yad'], ['ac', 'yc'], 'max_intensity')
pvals['pval_mean_AD/Control'] = compute_pvals(df, ['aad', 'yad'], ['ac', 'yc'], 'mean_intensity')
pvals['pval_max_tic_AD/Control'] = compute_pvals(df, ['aad', 'yad'], ['ac', 'yc'], 'max_tic_intensity')
pvals['pval_mean_tic_AD/Control'] = compute_pvals(df, ['aad', 'yad'], ['ac', 'yc'], 'mean_tic_intensity')

pvals['pval_max_Aged/Young'] = compute_pvals(df, ['aad', 'ac'], ['yad', 'yc'], 'max_intensity')
pvals['pval_mean_Aged/Young'] = compute_pvals(df, ['aad', 'ac'], ['yad', 'yc'], 'mean_intensity')
pvals['pval_max_tic_Aged/Young'] = compute_pvals(df, ['aad', 'ac'], ['yad', 'yc'], 'max_tic_intensity')
pvals['pval_mean_tic_Aged/Young'] = compute_pvals(df, ['aad', 'ac'], ['yad', 'yc'], 'mean_tic_intensity')

# 6️⃣ Combine into final dataframe
final_df = pd.DataFrame({'common_group': pivot_max.index})

for k, v in fold_changes.items():
    final_df[k] = v.values

for k, v in pvals.items():
    final_df[k] = v.values

final_df.head()


,common_group,fc_max_aad/yad,fc_max_aad/ac,fc_max_yad/yc,fc_mean_aad/yad,fc_mean_aad/ac,fc_mean_yad/yc,fc_max_tic_aad/yad,fc_max_tic_aad/ac,fc_max_tic_yad/yc,...,pval_mean_tic_aad/ac,pval_mean_tic_yad/yc,pval_max_AD/Control,pval_mean_AD/Control,pval_max_tic_AD/Control,pval_mean_tic_AD/Control,pval_max_Aged/Young,pval_mean_Aged/Young,pval_max_tic_Aged/Young,pval_mean_tic_Aged/Young
0,326.1952,0.717696,1.234669,1.079034,0.573082,1.469853,1.213358,0.492992,0.722410,1.142294,...,3.140248e-02,0.113906,0.257787,0.183548,0.769050,0.797178,2.091109e-06,0.000095,2.551810e-05,0.000261
1,337.1898,0.556273,1.385309,0.844522,0.524370,1.550963,0.883249,0.414081,0.887695,0.894250,...,8.454460e-02,0.597370,0.933513,0.888705,0.618733,0.655357,3.103893e-08,0.000012,8.706806e-09,0.000018
2,339.2275,1.249037,1.239592,1.311598,1.130248,1.167675,1.517534,0.767790,0.775848,1.454886,...,2.002993e-02,0.014203,0.015256,0.011890,0.590005,0.560189,1.074597e-02,0.037718,6.787419e-01,0.037687
3,340.2314,1.143730,1.145127,1.274712,1.154189,1.198580,1.502613,0.767483,0.738786,1.441095,...,2.025894e-02,0.015421,0.017219,0.009674,0.844043,0.596903,1.862171e-02,0.035544,5.203384e-01,0.031149
4,341.2417,1.403605,1.821431,0.913137,1.711283,2.844526,0.819830,0.978568,1.092022,1.053168,...,7.560248e-07,0.091583,0.021469,0.032277,0.024067,0.116598,7.612403e-01,0.838545,2.334852e-01,0.634700


In [4]:
final_df.to_csv(os.path.join(csv_folder_path, csv_output_name), index=False)